# Recuperação e limiar

Notebook de análise de artefatos: não inicia downloads ou treinamento. Não substitui revisão humana de fundamentação.

## Método
TF-IDF combina palavras e caracteres; a busca restringe fontes locais ao município declarado e retorna até três trechos. Limiar escolhido somente na validação. Abaixo, as métricas são recalculadas sem modificar o limiar congelado. Recall de busca e capacidade de responder são medidas diferentes; a calibração de respondibilidade é uma aproximação limitada.

In [1]:
import json
from pathlib import Path
r=json.loads(Path('configs/retrieval.json').read_text()); print({k:v for k,v in r.items() if k!='records'})

{'threshold': 0.16, 'f1': 0.47619047619047616, 'tp': 20, 'fp': 44, 'fn': 0, 'metrics_valid': {'recall@1': 0.5, 'recall@3': 0.5375, 'hit@1': 0.5, 'hit@3': 0.5375}, 'base_sha': '293f559b96abd075ae50158d866a94e05f6a882a51cb5910544273449212916f', 'valid_sha': 'd105ace92684a66998c98061bf4fe0be1cfef6cc838b17c60b32df7182eb1435'}


In [2]:
print('As métricas de recuperação não são acerto da geração. O limiar usa somente validação.')

As métricas de recuperação não são acerto da geração. O limiar usa somente validação.


In [3]:
from citizentech_ia.pipeline import load_retriever
from citizentech_ia.inference import retrieval_query
from citizentech_ia.schemas import Occurrence
from citizentech_ia.datasets import read_jsonl
from citizentech_ia.retrieval import retrieval_metrics
retriever=load_retriever(Path('.'))
records=[]
for row in read_jsonl('dados/valid.jsonl'):
 occurrence=Occurrence.model_validate(row['input'])
 hits=retriever.search(retrieval_query(occurrence),municipality=occurrence.municipio,threshold=r['threshold'])
 records.append({'categoria':row['category'],**retrieval_metrics([h['chunk_id'] for h in hits],row['relevant_chunks'])})
for key in ['recall@1','recall@3']:
 values=[x[key] for x in records if x['categoria']=='respondivel' and x[key] is not None]
 print('Validação respondível', key, sum(values)/len(values) if values else None)

Validação respondível recall@1 1.0
Validação respondível recall@3 1.0
